In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from matplotlib import rcParams
from matplotlib.colors import LogNorm
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
rcParams.update({'figure.autolayout': True})
import os
import IPython

In [24]:
GRAV = 4 * np.pi**2 #AU^3/M_sun/yr^2

#acceleration due to gravity
def g_accel(m,x,y):
    r =  np.sqrt(x**2 + y**2)
    GM = GRAV * m
    ax = -GM * x / r**3
    ay = -GM * y / r**3
    return ax, ay


In [32]:
#parameters to change
theta_e = np.pi *0   #earth position in radian
dt = 0.01 #year
total_t = 3 #year


#initial conditions
M_sun  = 1 #M_sun
steps = int(total_t / dt)
t = np.arange(steps + 1) * dt


#parameter for Eulers Method
x_e = np.zeros(steps + 1)
y_e = np.zeros(steps + 1)
vx_e = np.zeros(steps + 1)
vy_e = np.zeros(steps + 1)

#initial value
x_e[0] = 1* np.cos(theta_e)#AU
y_e[0] = 1* np.sin(theta_e)
vx_e[0] = -2 * np.pi * np.sin(theta_e)
vy_e[0] = 2 * np.pi * np.cos(theta_e)#AU/year

#Eulers Method
for n in range(steps):
    ax,ay = g_accel(M_sun,x_e[n],y_e[n])

    vx_e[n+1] = vx_e[n] + dt * ax
    vy_e[n+1] = vy_e[n] + dt * ay

    x_e[n+1] = x_e[n] + dt * vx_e[n] #using old value of v per Euler
    y_e[n+1] = y_e[n] + dt * vy_e[n]
      

#some other parameters of interest
speed_e = np.sqrt(vx_e**2 + vy_e**2)
r_e = np.sqrt(x_e**2 + y_e**2)
energy_e = 0.5 * speed_e**2 - GRAV*M_sun / r_e

In [33]:
#parameter for Leapfrog Method
x_l = np.zeros(steps + 1)
y_l = np.zeros(steps + 1)
vx_l = np.zeros(steps + 1)
vy_l = np.zeros(steps + 1)

#initial value
x_l[0] = 1* np.cos(theta_e)#AU
y_l[0] = 1* np.sin(theta_e)
vx_l[0] =  -2 * np.pi * np.sin(theta_e)
vy_l[0] =2 * np.pi * np.cos(theta_e)#AU/year
#Leapfrog method
for n in range(steps):
    ax,ay = g_accel(M_sun,x_l[n],y_l[n])

    # kick: half-step velocity
    vx_half = vx_l[n] + 0.5 * dt * ax
    vy_half = vy_l[n] + 0.5 * dt * ay

    # drift: full-step position
    x_l[n+1] = x_l[n] + dt * vx_half
    y_l[n+1] = y_l[n] + dt * vy_half

    # acceleration at new position
    ax_new, ay_new = g_accel(M_sun,x_l[n+1], y_l[n+1])

    # kick: finish velocity step
    vx_l[n+1] = vx_half + 0.5 * dt * ax_new
    vy_l[n+1] = vy_half + 0.5 * dt * ay_new

speed_l = np.sqrt(vx_l**2 + vy_l**2)
r_l = np.sqrt(x_l**2 + y_l**2)
energy_l = 0.5 * speed_l**2 - GRAV*M_sun / r_l

In [31]:
#learning from Matplotlib's animation page
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_aspect('equal')
ax.set_xlim(-5,5)
ax.set_ylim(-5,5)
ax.set_xlabel("x [AU]",fontsize = 20)
ax.set_ylabel("y [AU]",fontsize = 20)
ax.set_title("Earth's Orbit",fontsize = 25)

#objects
sun, = ax.plot(0, 0, 'o', c= "red" ,markersize=10, label='Sun')
time = ax.text(0.02, 0.95, '', transform=ax.transAxes,fontsize = 15)

trail_e, = ax.plot(x_e[0], y_e[0], '-', c = "green", lw=1, label = 'Euler')
earth_e, = ax.plot(x_e[0], y_e[0], 'o', c = "blue", markersize=6)

trail_l, = ax.plot(x_l[0], y_l[0], '-', c= "orange", lw=1, label = 'Leapfrog')
earth_l, = ax.plot(x_l[0], y_l[0], 'o', c = "blue", markersize=6)
ax.legend()


def update(frame):
    earth_e.set_data([x_e[frame]], [y_e[frame]])
    trail_e.set_data(x_e[:frame+1], y_e[:frame+1])

    earth_l.set_data([x_l[frame]], [y_l[frame]])
    trail_l.set_data(x_l[:frame+1], y_l[:frame+1])

    time.set_text(f"t = {t[frame]:.2f} yr")
    return earth_e, trail_e, time, earth_l, trail_l

ani = FuncAnimation(fig,update,frames=len(t),interval=30)

plt.close(fig) 
ani.save("earth_orbit.gif", writer="pillow", fps=20)
HTML(ani.to_jshtml())